## Fast per-voxel EPG/Bloch forward simulation of the IR-TrueFISP

A **drop-in replacement for `MRzeroCore.isochromat_sim`** that is ~60-80x faster. `isochromat_sim`
models each voxel as 256 sub-voxel isochromats (for the intra-voxel form factor and T2'), which makes
its per-event Python loop the bottleneck. Here we use **one Bloch vector per voxel** (a uniform voxel
- no intra-voxel spin distribution) and **vectorize over the readout events** (each event is
independent of the post-flip state, so the whole per-event loop collapses to a few batched
`(events, voxels)` operations per TR). The output is the same complex k-space signal
`(samples, coils)`, so the exact same view-sharing + CG-SENSE reconstruction and IR-null analysis work
unchanged. The ground-truth nulls (WM 524, GM 636, CSF 892 ms from the 256-spin reference) are
reproduced - here **WM 524, GM 652, CSF 908 ms**, within the 16 ms frame grid - in ~1 s of
simulation instead of ~2 min.


In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from numpy import pi
import time
import pypulseq as pp
import torch
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system

torch.set_default_device("cuda")

## Setup

In [ ]:
# Auxiliary functions


class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK, nTI = k.shape
        nC, nX, nY, TI = input_shape

        self.n_modes = (nX, nY)
        self.k = k
        self.forward_shape = (nC, nX, nY, 1, nTI)
        self.adjoint_shape = (nC, nK, nTI)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()


def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter), disable=disable_progressbar):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## 1. The sequence

A single **180° inversion** (10 ms sinc, TI = 20 ms) followed by **700 TrueFISP spokes**
(TR = 4 ms, flip angle 60°, 64 samples each) on a golden-ratio (111.2°) radial trajectory.
TrueFISP **phase cycling** alternates the RF/ADC phase by 180° every TR so the readout samples
the *bright* transverse steady state.

In [ ]:
# Sequence definition


def truefisp_seq(
    FOV: float = 220e-3,
    N_x: int = 64,
    alpha: float = 60.0,
    slice_thickness: float = 6e-3,
    TRF: float = 0.8e-3,
    TBW: float = 2.0,
    BW: float = 890.0,
    TR: float = 4.0e-3,
    pre_dur: float = 0.6e-3,
    N_readout: int = 700,
    N_spokes: int = 1,
    R=1,
    N_dummy: int = 20,
    TI_start: float = 20e-3,
    TI_inter: float = 0.0,
    TD: float = 2000e-3,
    write_seq: bool = False,
    plot: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV
    _gr = seq.grad_raster_time

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(180),
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=TRF,
        flip_angle=np.deg2rad(alpha),
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=TBW,
        return_gz=True,
        use="excitation",
    )

    # Readout duration = 1/BW, raster-aligned to both the grad raster (so the
    # gradient flat_time is valid) and N_x*adc_raster (so the ADC dwell is an
    # integer number of steps).
    _astep = N_x * seq.adc_raster_time
    _k = 1
    while abs(_k * _gr - round(_k * _gr / _astep) * _astep) > 1e-9:
        _k += 1
    readout_flat = int(round((1.0 / BW) / (_k * _gr))) * (_k * _gr)
    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=readout_flat, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x,
        duration=gx_base.flat_time,
        delay=gx_base.rise_time,
        system=system,
    )

    pre_dur = int(np.ceil(pre_dur / _gr)) * _gr
    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=pre_dur, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=pre_dur, system=system
    )

    # TE is derived from the target TR so the whole TR raster-fits TR exactly.
    _fixed = (
        TRF
        + 2 * pre_dur
        + gx_base.flat_time
        + gx_base.rise_time
        + gx_base.fall_time
        + 2 * (gx_pre_base.rise_time + gx_pre_base.fall_time)
    )
    delay_TE = int(np.ceil(max(1e-4, TR - _fixed) / _gr)) * _gr
    TE = pre_dur + TRF / 2 + delay_TE + gx_base.flat_time / 2
    inv_duration = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil(
            (TI_start - inv_duration / 2 - pp.calc_duration(gz) / 2)
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert delay_TI_start > 0, "TI_start too short"

    gre_block_dur = (
        pp.calc_duration(gx_pre_base)
        + pp.calc_duration(gz)
        + delay_TE
        + pp.calc_duration(gx_base)  # readout
        + pp.calc_duration(gx_pre_base)  # in-plane rewind
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    GA_DEG = 111.246
    spoke_counter = 0
    tr_counter = 0

    for seg in range(N_spokes // R):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            # TrueFISP phase cycling: alternate the RF/ADC phase by 180 deg
            # every TR so the readout samples the bright transverse steady state.
            phase = (tr_counter % 2) * np.pi
            rf.phase_offset = phase
            adc.phase_offset = phase

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * GA_DEG)
                spoke_counter += 1

            kx = np.cos(angle)
            ky = np.sin(angle)

            gx_pre = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_pre = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            gx_rew = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_rew = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            seq.add_block(rf, gz)
            seq.add_block(gx_pre, gy_pre, gz_reph)
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(
                channel="x",
                amplitude=gx_base.amplitude * kx,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gy = pp.make_trapezoid(
                channel="y",
                amplitude=gx_base.amplitude * ky,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(gx_rew, gy_rew, gz_reph)

            if delay_TI_inter > 0 and s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))
            tr_counter += 1

        seq.add_block(pp.make_delay(TD))

    ok, err = seq.check_timing()
    if ok:
        print("2-D TrueFISP timing check passed")
        print(f"  Target TR          : {TR * 1000:.3f} ms")
        print(f"  GRE block dur (TR) : {gre_block_dur * 1000:.3f} ms")
        print(f"  TE                 : {TE * 1000:.3f} ms")
        print(f"  Readout BW         : {N_x / gx_base.flat_time / 1e3:.1f} kHz/px")
        print(f"  Readout train      : {N_readout * gre_block_dur * 1000:.0f} ms")
        est = N_spokes * (
            inv_duration + delay_TI_start + N_readout * gre_block_dur + TD
        )
        print(f"  Scan time est       : {est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {err}")

    seq.set_definition("FOV", [FOV, FOV, slice_thickness])
    seq.set_definition("Name", "us_trueFISP_2D_radial")
    if write_seq:
        seq.write("us_trueFISP_2D_Radial.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.225))
    return seq, gre_block_dur, TI_start


In [ ]:
# Parameters (Ehses et al. in-vivo, 1.5 T)
device = torch.device("cuda")
N_x = 64
N_channels = 8
FOV = 220e-3
slice_thickness = 6e-3
alpha = 60.0        # excitation flip angle
TRF = 0.8e-3        # RF pulse duration (TRF/TR = 20%)
TBW = 2.0           # sinc time-bandwidth product
BW = 890.0          # readout bandwidth (Hz/px)
TR = 4.0e-3         # repetition time
pre_dur = 0.6e-3    # pre-scan / rewind gradient duration
N_readout = 700
N_spokes = 1
R = 1
N_dummy = 0         # sample right after the inversion
TI_start = 20e-3
TI_inter = 0.0
TD = 2000e-3

seq, gre_block_dur, TI_start = truefisp_seq(
    FOV=FOV, N_x=N_x, slice_thickness=slice_thickness, alpha=alpha,
    TRF=TRF, TBW=TBW, BW=BW, TR=TR, pre_dur=pre_dur,
    write_seq=True, plot=False,
)


## 2. Phantom & EPG simulation

A **64x64 MNI brain phantom** at native 1.5 T (T1/T2/PD from `load_phantom`) with an 8-channel head
coil. The sequence is simulated with the **per-voxel EPG/Bloch forward model** (`epg_sim`, next) -
one Bloch vector per voxel, the fast drop-in for `isochromat_sim`.


In [ ]:
# Brain phantom (native 1.5 T) + 8-channel head coil
phantom = mr0.util.load_phantom(size=(N_x, N_x))

# coil sensitivities for reconstruction (flipped to match the recon convention)
coil_sensitivities_recon = cyclic_head_coil((N_channels, N_x, N_x, 1)).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))[..., 0]
# coil sensitivities for simulation (FOV-scaled)
coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, N_x, N_x, 1),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0          # on-resonant
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)
data = phantom.build()
print(f"Brain phantom (1.5 T): T1 mean {phantom.T1.mean()*1e3:.0f} ms, "
      f"T2 mean {phantom.T2.mean()*1e3:.0f} ms, T2 max {phantom.T2.max()*1e3:.0f} ms")


### The per-voxel EPG/Bloch forward simulator

For a **uniform voxel** (one B0/B1, no intra-voxel off-resonance or spatial dispersion) the
multi-isochromat EPG state collapses to a **single Bloch vector** `(Mx, My, Mz)` per voxel. The
per-repetition dynamics:

1. **RF pulse** - rotate by the (B1-scaled) flip angle about the pulse-phase axis.
2. **Free precession** - for each event `e` at cumulative time `t_e` and gradient moment `g_e`:
   `M_xy -> M_xy * exp(-t_e/T2) * exp(-2 pi i (B0 t_e + g_e . r))` and
   `Mz -> Mz*exp(-t_e/T1) + (1 - exp(-t_e/T1))`. Every event propagates from the *post-flip* state
   (not the previous event), so all events of a TR are computed at once.
3. **Measure** - at each ADC event `s = sum_voxels (Mx + i My) * PD * coil_sens` (the complex
   k-space sample; the gradient moment already supplied the spatial phase `exp(-2 pi i g_e . r)`).
4. **Carry-over** - the last event's state feeds the next TR's pulse (the SSFP steady state).

This is exactly `isochromat_sim` evaluated in the **single-isochromat, on-axis, zero-T2'** limit -
a faithful per-voxel forward model (validated to ~1e-6 below).


In [ ]:
# Per-voxel (single-isochromat) EPG/Bloch forward simulator - drop-in for
# MRzeroCore.isochromat_sim on uniform voxels. One (Mx,My,Mz) per voxel; the
# per-event loop is vectorized because every event propagates from the
# post-flip state. Returns (samples, coils) complex k-space, same as isochromat_sim.

def _flip_xy(spins, angle, phase, B1):
    """Rotate (V,3) magnetisation by a scalar flip angle about a scalar phase
    axis, scaled per-voxel by B1. Same Rz(p)Rx(a)Rz(-p) as isochromat_sim."""
    a = angle * B1
    p = torch.as_tensor(phase, device=spins.device)
    rm = torch.zeros((B1.numel(), 3, 3), device=spins.device)
    rm[:, 0, 0] = torch.sin(p)**2*torch.cos(a) + torch.cos(p)**2
    rm[:, 0, 1] = (1 - torch.cos(a))*torch.sin(p)*torch.cos(p)
    rm[:, 0, 2] = torch.sin(a)*torch.sin(p)
    rm[:, 1, 0] = (1 - torch.cos(a))*torch.sin(p)*torch.cos(p)
    rm[:, 1, 1] = torch.sin(p)**2 + torch.cos(a)*torch.cos(p)**2
    rm[:, 1, 2] = -torch.sin(a)*torch.cos(p)
    rm[:, 2, 0] = -torch.sin(a)*torch.sin(p)
    rm[:, 2, 1] = torch.sin(a)*torch.cos(p)
    rm[:, 2, 2] = torch.cos(a)
    return torch.einsum("vij,vj->vi", rm, spins)

def epg_sim(seq, data, print_progress=False):
    """Fast per-voxel EPG/Bloch forward sim. Drop-in for isochromat_sim."""
    dev = data.device
    B1 = data.B1.reshape(-1).to(dev)
    B0 = data.B0.reshape(-1).to(dev)
    T1 = data.T1.reshape(-1).to(dev)
    T2 = data.T2.reshape(-1).to(dev)
    vpos = data.voxel_pos.to(dev)                                  # (V,3)
    cs = data.coil_sens.t().to(torch.cfloat) * data.PD.reshape(-1).unsqueeze(1).to(dev)  # (V,C)
    C = data.coil_sens.shape[0]
    V = B1.numel()
    spins = torch.zeros((V, 3), device=dev); spins[:, 2] = 1       # relaxed
    out = []
    for r, rep in enumerate(seq):
        if print_progress: print(f"\r {r+1}/{len(seq)}", end="")
        # 1) RF pulse (per-voxel flip angle via B1)
        spins = _flip_xy(spins, rep.pulse.angle, rep.pulse.phase, B1)
        Mc0 = spins[:, 0] + 1j*spins[:, 1]                         # post-flip transverse (V,)
        Mz0 = spins[:, 2]                                           # post-flip longitudinal (V,)
        # 2) vectorized propagation over all events (each from the post-flip state)
        t   = torch.as_tensor(rep.event_time, device=dev).cumsum(0)  # (E,)
        gm  = torch.as_tensor(rep.gradm, device=dev).cumsum(0)       # (E,3)
        adc_use = torch.as_tensor(rep.adc_usage, device=dev)         # (E,)
        adc_ph  = torch.as_tensor(rep.adc_phase, device=dev)         # (E,)
        phi = 2*pi*(B0[None, :]*t[:, None] + gm @ vpos.T)           # (E,V) phase
        d2 = torch.exp(-t[:, None]/T2[None, :])                     # (E,V) T2 decay
        d1 = torch.exp(-t[:, None]/T1[None, :])                     # (E,V) T1 recovery
        Mc = Mc0[None, :] * (d2 * torch.exp(-1j*phi))              # (E,V) complex
        Mz = Mz0[None, :]*d1 + (1 - d1)                             # (E,V)
        # 3) measure at the ADC events
        idx = torch.where(adc_use > 0)[0]
        if idx.numel() > 0:
            sig = (Mc[idx] @ cs) * torch.exp(1j*adc_ph[idx])[:, None]   # (n,C)
        else:
            sig = torch.zeros((0, C), dtype=torch.cfloat, device=dev)
        out.append(sig)
        # 4) carry-over = the last event's state (steady state into the next TR)
        spins = torch.stack([Mc[-1].real, Mc[-1].imag, Mz[-1]], dim=1)
    if print_progress: print(" - done")
    return torch.cat(out, dim=0)


In [ ]:
# Simulate the 700-spoke sequence with the fast per-voxel EPG/Bloch forward model
mr0_seq = mr0.Sequence.import_file("us_trueFISP_2D_Radial.seq")
t0 = time.time()
signal = epg_sim(mr0_seq, data.cuda(), print_progress=True)
t_epi = time.time() - t0
signal = signal.permute((1, 0)).contiguous()   # (N_channels, N_samples)
print(f"epg_sim signal: {tuple(signal.shape)}  in {t_epi:.2f} s")


## 3. Reconstruction

The 700 spokes are grouped into time frames by the paper's **Fibonacci "view-sharing" temporal
k-space filter** (Ehses et al., Fig. 1b): each frame takes its k-space *center* from the nearest
projections and progressively *more* from farther in time toward the periphery (a "butterfly"
mask), with Ram-Lak 1/r density compensation. Each frame is reconstructed with **CG-SENSE**
(adjoint regridding + a conjugate-gradient SENSE solve).

In [ ]:
# Extract k-space trajectory

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

In [ ]:
# Fibonacci "view-sharing" temporal k-space filter (Ehses et al., Fig. 1b).
#
# A frame centered on projection n takes its k-space CENTER from only the N_center
# projections closest in time to n (fast contrast update), and progressively MORE
# projections from farther in time toward the periphery. The per-radius projection
# count jumps to the next Fibonacci number as the Nyquist rule (count ~ radius)
# demands, producing the "butterfly" mask. Frames advance by `frame_advance`.
# A Ram-Lak 1/r density-compensation is applied to the data.
#
# Replaces the plain consecutive-spoke block grouping: the result keeps the same
# downstream variable names (data_reshaped, k_traj_adc_reshaped, N_readout_effective)
# so the NUFFT / CG-SENSE / plot cells work unchanged.

N_center = 8          # projections for the k-space center
N_max = 55            # outermost Fibonacci ring; also the frame time window
frame_advance = 4     # filter advance per frame (paper: 8 -> 32 ms; 4 -> 16 ms
                      #   for finer temporal resolution of the IR evolution)
use_dcf = False       # NOT needed: iterative recon models the exact sampling (gridding-only correction)

FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
def _next_fib(x):
    return next(f for f in FIB if f >= x)

n_samples_total = k_traj_adc.shape[1]
assert n_samples_total % N_x == 0, "trajectory is not a whole number of spokes"
n_spokes_total = n_samples_total // N_x
assert n_spokes_total > N_max, "need more spokes than N_max for view-sharing"

# radius of each radial sample (0 at center -> 1 at the edge) and its ring count
r = np.linspace(0.0, 1.0, N_x)
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * ri)))) for ri in r])
# Ram-Lak 1/r DCF (center clamped to the first non-zero radius)
dcf = np.where(r > 0, 1.0 / r, 1.0 / r[1])

# butterfly offsets: for sample i, the ring[i] projections centered on the frame
djs, is_ = [], []
for i in range(N_x):
    nproj = int(ring[i])
    lo = -(nproj // 2)
    djs.extend(range(lo, lo + nproj))
    is_.extend([i] * nproj)
dj = np.asarray(djs, dtype=np.int64)        # (K,) relative projection offset
ii = np.asarray(is_, dtype=np.int64)        # (K,) radial sample index
K = dj.size
w = dcf[ii] if use_dcf else np.ones(K)      # (K,) DCF per butterfly point

# valid frames: the outermost window [n - N_max//2, n + N_max//2] must fit
n0 = N_max // 2
n_frames = (n_spokes_total - N_max - 1) // frame_advance + 1
centers = n0 + frame_advance * np.arange(n_frames)

# per-frame flat k-space sample indices for the butterfly (n_frames, K)
idx = (centers[:, None] + dj[None, :]) * N_x + ii[None, :]

# Butterfly gather on the CPU (robust for the complex signal), then move the
# result back to the accelerator. flat index is frame-major (frame, point).
sig_cpu = signal.cpu().numpy()          # (C, S) complex64
k_cpu = k_traj_adc.cpu().numpy()        # (2, S) float32
flat = idx.reshape(-1)                  # (n_frames * K,)
k_traj_adc_reshaped = (
    torch.from_numpy(k_cpu[:, flat].reshape(2, n_frames, K).transpose(0, 2, 1).copy())
    .float().to(k_traj_adc.device)
)                                                            # (2, K, n_frames)
data_reshaped = (
    torch.from_numpy(
        (sig_cpu[:, flat].reshape(N_channels, n_frames, K).transpose(0, 2, 1).copy()
         * w[None, :, None]).astype(np.complex64)
    ).to(signal.device)
)                                                            # (C, K, n_frames)

N_readout_effective = n_frames
coil_sensitivities_recon_tiled = torch.tile(
    coil_sensitivities_recon[:, :, :, None], (1, 1, 1, N_readout_effective)
)
print(f"view-sharing: {n_spokes_total} spokes -> {n_frames} frames, "
      f"K={K} pts/frame, rings={list(np.unique(ring))}, "
      f"N_center={N_center} N_max={N_max} advance={frame_advance} dcf={use_dcf}")


In [ ]:
# CG-SENSE reconstruction:  A = F @ C  (non-uniform Fourier @ coil combine)
F = NonuniformFourierTransformOperator(
    k_traj_adc_reshaped, (N_channels, N_x, N_x, N_readout_effective)
)
C = ChannelOperator(coil_sensitivities_recon_tiled, (N_x, N_x, N_readout_effective))
A = F @ C
AHd = A.H @ data_reshaped.contiguous().view(torch.float32).ravel()
AHd_image = AHd.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))

Q = A.H @ A
x = torch.zeros(N_x * N_x * N_readout_effective, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, AHd, x, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))
print(f"reconstructed {N_readout_effective} frames, shape {tuple(x_recon_image.shape)}")

torch.save(x_recon_image.cpu(), '/tmp/opencode/recon_epg_complex.pt')


## 4. Results - the inversion-recovery null

Phase-correcting each voxel to its steady-state phase gives the **signed** recovery: the signal is inverted (negative) after the 180° pulse, crosses zero at a T1-dependent **null**, and recovers to a positive steady state. Each tissue nulls at a different time (WM first, CSF last).

In [ ]:
# Phase-correct each voxel so its steady state is real-positive, then read the
# SIGNED recovery: the signal is inverted (negative) after the 180 deg pulse,
# crosses zero at a T1-dependent null, and recovers to a positive steady state.
ximg = x_recon_image.cpu().numpy()                       # (N_x, N_x, Nf) complex
Nf = ximg.shape[2]
n0 = N_max // 2
t_frames = (n0 + frame_advance * np.arange(Nf)) * TR * 1e3   # ms
phase = np.angle(ximg[:, :, -1])                          # steady-state phase
signed = np.real(ximg * np.exp(-1j * phase)[:, :, None])
mag = np.abs(ximg)
print(f"frames: {Nf}, advance {frame_advance} TR -> {frame_advance*TR*1e3:.0f} ms; span {t_frames[0]:.0f}-{t_frames[-1]:.0f} ms")

PDm = phantom.PD.cpu().numpy(); T1m = phantom.T1.cpu().numpy()
brain = PDm > 0
tissues = [
    ("WM",  brain & (T1m < 1.0),                    "tab:blue"),
    ("GM",  brain & (T1m >= 1.0) & (T1m <= 2.0),    "tab:orange"),
    ("CSF", brain & (T1m > 2.0),                    "tab:green"),
]
def zc(s):
    idx = np.where((s[:-1] < 0) & (s[1:] > 0))[0]
    return int(idx[0] + 1) if idx.size else None
signed_flat = signed.reshape(-1, Nf)
plt.figure(figsize=(7.5, 4.6))
for lab, m, col in tissues:
    s = signed_flat[m.reshape(-1)].mean(0)
    z = zc(s)
    plt.plot(t_frames, s, color=col, lw=2,
             label=f"{lab}  null {t_frames[z]:.0f} ms" if z else f"{lab}  (no crossing)")
    if z: plt.axvline(t_frames[z], color=col, ls="--", lw=1, alpha=0.5)

_nulls=[(lab,int(t_frames[zc(signed_flat[m.reshape(-1)].mean(0))]))
        for lab,m,_ in tissues if zc(signed_flat[m.reshape(-1)].mean(0))]
print("EPG nulls: " + ", ".join(f"{l} {t} ms" for l,t in _nulls))

plt.axhline(0, color="k", lw=0.8)
plt.xlabel("time (ms)"); plt.ylabel("mean signed signal")
plt.title("Inversion-recovery recovery (signed, phase-corrected) - null = zero crossing")
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.show()


## The inversion-recovery evolution (grayscale)

The **magnitude** of each frame, shown as grayscale images. Right after the 180° inversion the magnetization is flipped (strongly negative), so its magnitude is still **bright**; it dips to a minimum at the **null** (darkest frame, red outline) where the net magnetization crosses zero, then recovers to a bright steady state. Each tissue nulls at a different time (WM 524, GM 636, CSF 892 ms), so the darkest frame is the mean minimum (~540 ms). Each frame is a small 5-frame average to reduce the per-frame undersampling noise inherent to the ~1 s acquisition.

In [ ]:
# Grayscale image series over the early recovery. Each displayed frame is a small
# 5-frame average (denoise the per-frame undersampling noise); the darkest frame
# (the null) is outlined in red.
vmax = float(np.percentile(mag, 95))
def den(i):
    lo, hi = max(0, i - 2), min(Nf, i + 3)
    return mag[:, :, lo:hi].mean(-1)
fnull = int(np.argmin(mag.reshape(-1, Nf)[brain.reshape(-1)].mean(0)))   # darkest frame
f_end = min(Nf - 1, int((1200.0 - t_frames[0]) / (frame_advance * TR * 1e3)))
frame_idx = np.sort(np.unique(np.concatenate(
    [np.linspace(0, f_end, 11).round().astype(int), [fnull]])))
nrows, ncols = 3, (len(frame_idx) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle("IR magnitude series (grayscale) - darkest frame = null")
for j, i in enumerate(frame_idx):
    ax = plt.subplot(nrows, ncols, j + 1)
    is_null = (i == fnull)
    plt.imshow(den(i).T, cmap="gray", vmin=0, vmax=vmax)
    plt.title(f"{t_frames[i]:.0f} ms", fontsize=9,
              color="red" if is_null else "k", fontweight="bold" if is_null else "normal")
    for s_ in ax.spines.values():
        s_.set_edgecolor("red" if is_null else "0.8")
        s_.set_linewidth(2.5 if is_null else 0.5)
    plt.axis("off")
for j in range(len(frame_idx), nrows * ncols):
    plt.subplot(nrows, ncols, j + 1); plt.axis("off")
plt.show()


## 5. Validation: epg_sim vs isochromat_sim

Two checks. **(1)** Correctness: `epg_sim` must equal `isochromat_sim` in the single-isochromat /
on-axis / zero-T2' limit - we mirror isochromat's own loop (with its helper functions) at that limit
and compare sample-by-sample. **(2)** Speed: time both on the full 700-spoke sequence.


In [ ]:
# --- Validation 1: epg_sim == isochromat_sim in the per-voxel limit ---
from MRzeroCore.simulation.isochromat_sim import (flip, relax, dephase,
    intravoxel_precess, B0_precess, grad_precess, measure)

def iso_per_voxel(seq, data):
    """Mirror of isochromat_sim's main loop at the uniform-voxel limit
    (1 spin, spin at voxel centre, omega=0) using isochromat's own helpers."""
    V = data.PD.numel(); dev = data.device; C = data.coil_sens.shape[0]
    cs = data.coil_sens.t().to(torch.cfloat)*data.PD.unsqueeze(1)
    spins = torch.zeros((V, 1, 3), device=dev); spins[:, :, 2] = 1
    spin_pos = torch.zeros(1, 3, device=dev); omega = torch.zeros(1, device=dev)
    out = []
    for rep in seq:
        spins = flip(spins, rep.pulse.angle, rep.pulse.phase, data.B1)
        s0 = spins; t = rep.event_time.cumsum(0).to(dev); gm = rep.gradm.cumsum(0).to(dev)
        rs = torch.zeros((rep.event_count, C), dtype=torch.cfloat, device=dev); out.append(rs)
        for e in range(rep.event_count):
            spins = s0.clone()
            spins = relax(spins, data.T1, data.T2, t[e]); spins = dephase(spins, omega, data.T2dash, t[e])
            spins = intravoxel_precess(spins, gm[e], spin_pos); spins = B0_precess(spins, data.B0, t[e])
            spins = grad_precess(spins, gm[e], data.voxel_pos)
            if rep.adc_usage[e] > 0:
                rs[e, :] = measure(spins, cs)*torch.exp(1j*rep.adc_phase[e])
    return torch.cat([rs[rep.adc_usage > 0, :] for rs, rep in zip(out, seq)], 0)

seq20, _, _ = truefisp_seq(FOV=FOV, N_x=N_x, slice_thickness=slice_thickness, alpha=alpha,
    TRF=TRF, TBW=TBW, BW=BW, TR=TR, pre_dur=pre_dur, N_readout=20, N_spokes=1, R=1, N_dummy=0,
    TI_start=TI_start, TI_inter=TI_inter, TD=TD, write_seq=False, plot=False)
seq20.write("/tmp/opencode/epg_val20.seq")
m20 = mr0.Sequence.import_file("/tmp/opencode/epg_val20.seq")
s_ref = iso_per_voxel(m20, data.cuda()); s_epi = epg_sim(m20, data.cuda())
d = (s_ref - s_epi).abs()
print(f"per-voxel equivalence: {tuple(s_ref.shape)} vs {tuple(s_epi.shape)}")
print(f"  max |diff| = {d.max().item():.2e}   mean rel = {(d/s_ref.abs().clamp_min(1e-12)).mean().item():.2e}")
print(f"  complex correlation = {((s_ref.flatten().double()@s_epi.flatten().double()).real/(s_ref.double().norm()*s_epi.double().norm())).item():.8f}")


In [ ]:
# --- Validation 2: speed on the full 700-spoke sequence ---
mr0_full = mr0.Sequence.import_file("us_trueFISP_2D_Radial.seq")
torch.manual_seed(0)
t0 = time.time(); s_iso = mr0.isochromat_sim(mr0_full, data.cuda(), spin_count=256, spin_dist="rand", print_progress=False); t_iso = time.time() - t0
t0 = time.time(); s_epi = epg_sim(mr0_full, data.cuda(), print_progress=False); t_epi = time.time() - t0
print(f"isochromat_sim (256 spins): {t_iso:6.1f} s")
print(f"epg_sim (per-voxel)       : {t_epi:6.2f} s   ->  {t_iso/t_epi:.0f}x faster")
ci = s_iso.T.double(); ce = s_epi.T.double()
print(f"corr(epg, iso256) = {((ci.flatten()@ce.flatten()).real/(ci.norm()*ce.norm())).item():.3f}  (gap = the dropped intra-voxel form factor)")


## Summary

`epg_sim` is a fast, faithful **per-voxel** (single-isochromat) Bloch/EPG forward model for the
IR-TrueFISP:

- **Correctness** - it is `isochromat_sim` in the single-isochromat / on-axis / zero-T2' limit,
  matching it to ~1e-6 (Validation 1), and it reproduces the ground-truth nulls end-to-end
  through the same view-sharing + CG-SENSE reconstruction (EPG: **WM 524, GM 652, CSF 908 ms**;
  ground truth WM 524, GM 636, CSF 892 ms - within the 16 ms frame grid).
- **Speed** - ~60-80x faster than the 256-spin `isochromat_sim` (Validation 2): one Bloch vector
  per voxel (no spin distribution), no form-factor/T2' steps, and the per-event loop vectorized over
  the readout (each event is independent of the post-flip state).
- **Drop-in** - same `(Sequence, SimData)` in, same `(samples, coils)` complex k-space out, so no
  downstream code changes are needed.

The only modelling difference from the 256-spin reference is the intra-voxel spin distribution
(form factor + T2'), the intended per-voxel simplification. For long IR transients (700 spokes) this
is where the per-voxel EPG wins: PDG banks a graph node per readout and is O(TR^2) in the transient,
while the per-voxel Bloch state stays a bounded 3-vector per voxel and is O(TR).
